First import modules and initialize Earth Engine.

In [ ]:

# standard modules
import io
import json
import os
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project='nr218-michaelhuggins')

Open the AOI vector file and check it on a map.

In [ ]:
# read AOI
# This works whether the notebook is launched from the repo root or from assets/.
this_dir = Path.cwd()
repo_dir = this_dir if (this_dir / 'assets').exists() else this_dir.parent

def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

aoi_path = first_existing_path([
    this_dir / 'gran_chaco_aoi.geojson',
    repo_dir / 'assets' / 'gran_chaco_aoi.geojson',
])

aoi_path


In [ ]:
aoi = gpd.read_file(aoi_path).to_crs(4326)

gee_json = json.loads(aoi[['geometry']].to_json())
gee_aoi = geemap.geojson_to_ee(gee_json)

# inspect the GeoJSON as an EEObject through geemap.
test_map = geemap.Map(basemap='SATELLITE')
test_map.centerObject(gee_aoi, 9)

aoi_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
test_map.addLayer(gee_aoi.style(**aoi_style), {}, 'Gran Chaco AOI')

test_map


Get the vertices of the Gran Chaco AOI to use in Earth Engine filters.


In [ ]:
# get extent for the AOI
def bounds_to_verts(gdf):
    minx, miny, maxx, maxy = gdf.total_bounds
    return [
        [float(minx), float(miny)],
        [float(minx), float(maxy)],
        [float(maxx), float(maxy)],
        [float(maxx), float(miny)],
        [float(minx), float(miny)],
    ]

verts = bounds_to_verts(aoi)
verts


# Dry-season composite periods

Use these periods to download annual dry-season composites for the Gran Chaco AOI.


In [ ]:
# Dry season in the Gran Chaco.
# End dates are exclusive, so August composites end on September 1.
DRY_START_MONTH = 6
DRY_END_MONTH = 8

dry_composite_periods = {
    'landsat_5_7': {
        'start_year': 2000,
        'end_year': 2011,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 30,
        'short_name': 'l57',
    },
    'landsat_8_9': {
        'start_year': 2013,
        'end_year': 2025,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 30,
        'short_name': 'l89',
    },
    'sentinel_2': {
        'start_year': 2018,
        'end_year': 2025,
        'start_month': DRY_START_MONTH,
        'end_month': DRY_END_MONTH,
        'scale': 10,
        'short_name': 's2',
    },
}

export_region = gee_aoi.geometry()


def dry_season_date_range(year):
    start = ee.Date.fromYMD(year, DRY_START_MONTH, 1)
    end = ee.Date.fromYMD(year, DRY_END_MONTH, 1).advance(1, 'month')
    return start, end


# Quick check: these are the image years that will be requested for each sensor group.
for sensor, period in dry_composite_periods.items():
    years = list(range(period['start_year'], period['end_year'] + 1))
    print(sensor, years[0], 'to', years[-1], f"({len(years)} composites)")


## Export dry-season annual composites to Google Drive.

Run the helper cell, then call `queue_dry_drive_exports('sentinel_2')` or another sensor key from `dry_composite_periods`.


In [ ]:
COMMON_BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
DRY_DRIVE_FOLDER = 'gran_chaco_dry_season'
LANDSAT_SUN_ELEVATION_MIN = 30
LANDSAT_SUN_ELEVATION_MAX = 55
S2_SOLAR_ZENITH_MIN = 35
S2_SOLAR_ZENITH_MAX = 60


def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = 1 << 4
    clouds = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(qa.bitwiseAnd(clouds).eq(0))
    return image.updateMask(mask)


def prep_landsat_5_7(image):
    optical = image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def prep_landsat_8_9(image):
    optical = image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        COMMON_BANDS,
    ).multiply(0.0000275).add(-0.2)
    return optical.copyProperties(image, image.propertyNames())


def mask_s2_clouds(image):
    qa = image.select('QA60')
    clouds = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(clouds).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    optical = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'], COMMON_BANDS)
    return optical.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())


def build_dry_season_composite(sensor, year, region):
    period = dry_composite_periods[sensor]
    start, end = dry_season_date_range(year)

    if sensor == 'landsat_5_7':
        collection = (
            ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.gte('SUN_ELEVATION', LANDSAT_SUN_ELEVATION_MIN))
            .filter(ee.Filter.lte('SUN_ELEVATION', LANDSAT_SUN_ELEVATION_MAX))
            .map(mask_landsat_sr)
            .map(prep_landsat_5_7)
        )
    elif sensor == 'landsat_8_9':
        collection = (
            ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.gte('SUN_ELEVATION', LANDSAT_SUN_ELEVATION_MIN))
            .filter(ee.Filter.lte('SUN_ELEVATION', LANDSAT_SUN_ELEVATION_MAX))
            .map(mask_landsat_sr)
            .map(prep_landsat_8_9)
        )
    elif sensor == 'sentinel_2':
        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35))
            .filter(ee.Filter.gte('MEAN_SOLAR_ZENITH_ANGLE', S2_SOLAR_ZENITH_MIN))
            .filter(ee.Filter.lte('MEAN_SOLAR_ZENITH_ANGLE', S2_SOLAR_ZENITH_MAX))
            .map(mask_s2_clouds)
        )
    else:
        raise ValueError(f'Unknown sensor: {sensor}')

    return collection.median().clip(region)


def queue_dry_drive_exports(sensor='sentinel_2', folder=DRY_DRIVE_FOLDER, region=export_region):
    period = dry_composite_periods[sensor]
    tasks = []

    for year in range(period['start_year'], period['end_year'] + 1):
        image = build_dry_season_composite(sensor, year, region)
        prefix = f"{period['short_name']}_{year}_dry_season"
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=prefix,
            folder=folder,
            fileNamePrefix=prefix,
            region=region,
            scale=period['scale'],
            fileFormat='GeoTIFF',
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)
        print(f'Started export: {prefix}')

    return tasks


# Example:
# dry_tasks = queue_dry_drive_exports('sentinel_2')


In [ ]:
tasks = []

for sensor in dry_composite_periods.keys():
    tasks.extend(queue_dry_drive_exports(sensor))

print(f'Queued {len(tasks)} export tasks.')

In [ ]:
flat_tasks = []
for task in tasks:
    if isinstance(task, list):
        flat_tasks.extend(task)
    else:
        flat_tasks.append(task)

seen_done = set()
terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}
not_done = True

while not_done:
    for task in flat_tasks:
        status = task.status()
        description = status.get('description', task.id)
        state = status.get('state', 'UNKNOWN')

        if state in terminal_states and description not in seen_done:
            print(description, state)
            seen_done.add(description)

    not_done = any(task.status().get('state') not in terminal_states for task in flat_tasks)

    if not_done:
        print('Waiting 3 minutes before checking again...')
        time.sleep(180)

print('All tasks are done.')